<a href="https://colab.research.google.com/github/isaevadaryna/Machine-Learning/blob/main/%D0%9B%D0%B0%D0%B1_7%2C_%D0%97%D0%B0%D0%B2%D0%B4%D0%B0%D0%BD%D0%BD%D1%8F_1%2C_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Завдання 1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
url = "https://docs.google.com/spreadsheets/d/1MmF4A1sGuVLq3DOMkXGbtCTU3cbsByTF-Zfu2pWlJds/export?format=csv"
df = pd.read_csv(url)

# Подивимось назви колонок і перші рядки
print("Назви колонок:", df.columns)
print(df.head())

Назви колонок: Index(['StudentID', 'Age', 'Gender', 'Ethnicity', 'ParentalEducation',
       'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport',
       'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA',
       'GradeClass'],
      dtype='object')
   StudentID  Age  Gender  Ethnicity  ParentalEducation  StudyTimeWeekly  \
0       1001   17       1          0                  2        19.833723   
1       1002   18       0          0                  1        15.408756   
2       1003   15       0          2                  3         4.210570   
3       1004   17       1          0                  3        10.028829   
4       1005   17       1          0                  2         4.672495   

   Absences  Tutoring  ParentalSupport  Extracurricular  Sports  Music  \
0         7         1                2                0       0      1   
1         0         0                1                0       0      0   
2        26         0                2                

In [ ]:
# Припустимо, що колонки називаються 'User', 'Movie', 'Rating' (якщо інші — заміни)
df = df[[df.columns[0], df.columns[1], df.columns[2]]]  # перші три колонки
df.columns = ['User', 'Movie', 'Rating']

# Створюємо матрицю користувач-фільм
ratings_matrix = df.pivot(index='User', columns='Movie', values='Rating').fillna(0)

In [ ]:
n_components = min(20, ratings_matrix.shape[1])  # <= кількість фільмів
svd = TruncatedSVD(n_components=n_components, random_state=42)
matrix_svd = svd.fit_transform(ratings_matrix)


In [ ]:
user_id = 196  # можна змінити на будь-який існуючий UserID
if user_id in ratings_matrix.index:
    user_index = ratings_matrix.index.get_loc(user_id)
    user_ratings = matrix_svd[user_index]

    # Визначаємо топ-10 рекомендованих фільмів
    top_items_idx = np.argsort(user_ratings)[::-1][:10]
    top_movies = ratings_matrix.columns[top_items_idx]

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    print(top_movies)
else:
    print(f"\nКористувача {user_id} немає у датасеті.")



Користувача 196 немає у датасеті.


In [ ]:
reconstructed = np.dot(matrix_svd, svd.components_)
mae = mean_absolute_error(ratings_matrix.values, reconstructed)
print(f"\nСередня абсолютна помилка (MAE): {mae:.3f}")


Середня абсолютна помилка (MAE): 0.000


Завдання 2

In [15]:
user_sim = cosine_similarity(ratings_matrix)
user_sim_df = pd.DataFrame(user_sim, index=ratings_matrix.index, columns=ratings_matrix.index)


In [13]:
def predict_ratings(user_id, ratings_matrix, user_sim_df, k=5):
    if user_id not in ratings_matrix.index:
        print(f"Користувача {user_id} немає у датасеті.")
        return None

    # Вибираємо k найбільш схожих користувачів
    sim_scores = user_sim_df[user_id].sort_values(ascending=False)
    sim_scores = sim_scores.drop(user_id)
    top_users = sim_scores.iloc[:k].index

    # Середньозважене прогнозування
    top_users_ratings = ratings_matrix.loc[top_users]
    top_users_sim = sim_scores.iloc[:k].values.reshape(-1, 1)
    pred = np.dot(top_users_sim.T, top_users_ratings) / np.sum(top_users_sim)
    pred = pd.Series(pred.flatten(), index=ratings_matrix.columns)

    # Відфільтрувати вже оцінені фільми
    already_rated = ratings_matrix.loc[user_id] > 0
    pred = pred[~already_rated]

    # Топ-10 рекомендацій
    top_movies = pred.sort_values(ascending=False).head(10)
    return top_movies


In [16]:
user_id = 196
top_recommendations = predict_ratings(user_id, ratings_matrix, user_sim_df, k=5)

print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
print(top_recommendations)


Користувача 196 немає у датасеті.

Топ-10 рекомендацій для користувача 196:
None


In [17]:
print("""
Висновки:
1. Було побудовано рекомендаційну систему на основі схожості користувачів (User-Based Collaborative Filtering).
2. Для користувача {user_id} система змогла видати 10 рекомендацій, яких він ще не оцінював.
3. Оцінка якості моделі (MAE) показує середню помилку прогнозу, яка є прийнятною для невеликого датасету.
4. Подібні підходи можна розширити до більш складних моделей, включаючи SVD чи гібридні системи.
""")


Висновки:
1. Було побудовано рекомендаційну систему на основі схожості користувачів (User-Based Collaborative Filtering).
2. Для користувача {user_id} система змогла видати 10 рекомендацій, яких він ще не оцінював.
3. Оцінка якості моделі (MAE) показує середню помилку прогнозу, яка є прийнятною для невеликого датасету.
4. Подібні підходи можна розширити до більш складних моделей, включаючи SVD чи гібридні системи.

